# Chapter 4: Loading Data, Evaluating Models, and Improving Performance

> Source PDF: `04_Loading_Data.pdf`  
> Instructor: Maham Faisal Khan, Senior Data Scientist

## Learning Objectives
By the end of this notebook, you will be able to:
- Load tabular data into features and targets.
- Wrap tensors with `TensorDataset` and `DataLoader`.
- Calculate training and validation loss.
- Track classification accuracy.
- Recognize and reduce overfitting.
- Use dropout and weight decay.
- Apply a practical performance improvement workflow.


In [ ]:
# Core imports used throughout this notebook
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split


## 4.1 Back to the Animals Dataset

The PDF uses an animals dataset with binary/categorical features and a target type.

| Type | Animal Group |
|---:|---|
| 1 | Mammal |
| 2 | Bird |
| 3 | Reptile |
| 4 | Fish |
| 5 | Amphibian |
| 6 | Bug |
| 7 | Invertebrate |


In [ ]:
# PDF snippet equivalent:
# animals = pd.read_csv('animals.csv')
# animals.head()

animals = pd.DataFrame({
    "animal_name": ["skimmer", "gull", "seahorse", "tuatara", "squirrel"],
    "hair":        [0, 0, 0, 0, 1],
    "feathers":    [1, 1, 0, 0, 0],
    "eggs":        [1, 1, 1, 1, 0],
    "milk":        [0, 0, 0, 0, 1],
    "predator":    [1, 1, 0, 1, 0],
    "fins":        [0, 0, 1, 0, 0],
    "legs":        [2, 2, 0, 4, 2],
    "tail":        [1, 1, 1, 1, 1],
    "type":        [2, 2, 4, 3, 1],
})

animals


## 4.2 Defining Features and Target Values

For supervised learning, **features** are the input columns and the **target** is the ground-truth value.


In [ ]:
import numpy as np

# Define input features
features = animals.iloc[:, 1:-1]
X = features.to_numpy()
print(X)


In [ ]:
# Define target feature (ground truth)
target = animals.iloc[:, -1]
y = target.to_numpy()
print(y)


## 4.3 Recalling `TensorDataset`

`TensorDataset` combines tensors so indexing returns aligned samples, such as `(features, label)`.


In [ ]:
import torch
from torch.utils.data import TensorDataset

# Instantiate dataset class
# CrossEntropyLoss expects integer class indices that start at 0, so subtract 1 from the PDF's type labels.
dataset = TensorDataset(
    torch.tensor(X).float(),
    torch.tensor(y - 1).long()
)

# Access an individual sample
sample = dataset[0]
input_sample, label_sample = sample
print("input sample:", input_sample)
print("label_sample:", label_sample)


## 4.4 Recalling `DataLoader`

A `DataLoader` iterates over a dataset in mini-batches and can shuffle examples at the start of each epoch.


In [ ]:
from torch.utils.data import DataLoader

batch_size = 2
shuffle = True

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


In [ ]:
# Iterate over the dataloader
for batch_inputs, batch_labels in dataloader:
    print("batch inputs", batch_inputs)
    print("batch labels", batch_labels)


## 4.5 Evaluating Model Performance

Raw datasets are usually split into three subsets.

| Subset | Typical Percent | Role |
|---|---:|---|
| Training | 80-90% | Adjust model parameters. |
| Validation | 10-20% | Tune hyperparameters. |
| Testing | 5-10% | Calculate final metrics once. |


In [ ]:
# A tiny train/validation split for demonstration
train_size = 4
validation_size = len(dataset) - train_size
train_dataset, validation_dataset = random_split(
    dataset,
    [train_size, validation_size],
    generator=torch.Generator().manual_seed(42)
)

trainloader = DataLoader(train_dataset, batch_size=2, shuffle=True)
validationloader = DataLoader(validation_dataset, batch_size=1, shuffle=False)

model = nn.Sequential(
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 7)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-2)


## 4.6 Calculating Training Loss

For each epoch, sum the loss for each training batch, then divide by the number of batches.


In [ ]:
training_loss = 0.0
model.train()

for i, data in enumerate(trainloader, 0):
    features_batch, labels = data

    # Run the forward pass
    outputs = model(features_batch)

    # Calculate the loss
    loss = criterion(outputs, labels)

    # Calculate the gradients and update parameters
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Calculate and sum the loss
    training_loss += loss.item()

epoch_loss = training_loss / len(trainloader)
print("Training loss:", epoch_loss)


## 4.7 Calculating Validation Loss

Validation should not update model weights. Use `model.eval()`, `torch.no_grad()`, and then `model.train()` before returning to training.


In [ ]:
validation_loss = 0.0
model.eval()  # Put model in evaluation mode

with torch.no_grad():  # Speed up the forward pass
    for i, data in enumerate(validationloader, 0):
        features_batch, labels = data

        # Run the forward pass
        outputs = model(features_batch)

        # Calculate the loss
        loss = criterion(outputs, labels)
        validation_loss += loss.item()

epoch_loss = validation_loss / len(validationloader)
print("Validation loss:", epoch_loss)
model.train()


## 4.8 Overfitting

Overfitting means the model memorizes training data and does not generalize well to unseen data.

| Symptom | Interpretation |
|---|---|
| Training loss improves, validation loss worsens | Model is overfitting. |
| Training accuracy high, validation accuracy poor | Model memorized training patterns. |


## 4.9 Calculating Accuracy

The PDF uses `torchmetrics.Accuracy`. The cell below tries to use `torchmetrics` when available and falls back to a manual calculation otherwise.


In [ ]:
try:
    import torchmetrics

    # Create accuracy metric using torchmetrics
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=7)
    for i, data in enumerate(dataloader, 0):
        features_batch, labels = data
        outputs = model(features_batch)
        # Calculate accuracy over the batch
        acc = metric(outputs, labels)

    # Calculate accuracy over the whole epoch
    acc = metric.compute()
    print(f"Accuracy on all data: {acc}")

    # Reset the metric for the next epoch
    metric.reset()
except ImportError:
    correct = 0
    total = 0
    with torch.no_grad():
        for features_batch, labels in dataloader:
            outputs = model(features_batch)
            predictions = outputs.argmax(dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    print(f"Accuracy on all data: {correct / total:.3f}")


## 4.10 Fighting Overfitting

| Problem | Solutions |
|---|---|
| Dataset is not large enough | Get more data or use data augmentation. |
| Model has too much capacity | Reduce model size or add dropout. |
| Weights are too large | Use weight decay. |


## 4.11 Regularization with Dropout

Dropout randomly zeroes elements of the input tensor during training. It is commonly added after an activation function.


In [ ]:
model = nn.Sequential(
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Dropout(p=0.5)
)

features = torch.randn((1, 8))
model.train()
print(model(features))


## 4.12 Regularization with Weight Decay

Weight decay adds a penalty to discourage large weights and biases.


In [ ]:
optimizer = optim.SGD(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(optimizer)


## 4.13 Data Augmentation

Data augmentation creates modified versions of training examples. It is especially common for images, audio, and text.

| Domain | Example Augmentation |
|---|---|
| Images | Crop, flip, rotate, color jitter |
| Audio | Add noise, time shift, speed perturbation |
| Text | Back-translation, token dropout, paraphrasing |


## 4.14 Improving Model Performance

The PDF recommends a three-step workflow:
1. Overfit the training set to prove the model and training loop can solve the task.
2. Reduce overfitting to improve validation performance.
3. Fine-tune hyperparameters using search.


In [ ]:
# Step 1: overfit a single training batch
model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 7)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

features_batch, labels = next(iter(trainloader))

for i in range(100):  # PDF shows range(1e3); 100 keeps the demo lightweight
    outputs = model(features_batch)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Final single-batch loss:", loss.item())
print("Predictions:", outputs.argmax(dim=-1).tolist())
print("Labels:", labels.tolist())


### Step 2: Reduce Overfitting

Experiment with dropout, data augmentation, weight decay, and reducing model capacity. Track every hyperparameter and report the maximum validation accuracy.


### Step 3: Fine-Tune Hyperparameters

Two simple search strategies are grid search and random search.


In [ ]:
# Grid search over learning-rate powers
for factor in range(2, 6):
    lr = 10 ** -factor
    print("Grid-search learning rate:", lr)

# Random search over the same exponent range
factor = np.random.uniform(2, 6)
lr = 10 ** -factor
print("Random-search learning rate:", lr)


## 4.15 Course Wrap-Up

| Chapter | What You Learned |
|---|---|
| Chapter 1 | Deep learning basics, small neural networks, linear layers, activations. |
| Chapter 2 | Loss functions, derivatives, backpropagation, and training loops. |
| Chapter 3 | Architecture, learning rate, momentum, and transfer learning. |
| Chapter 4 | DataLoaders, evaluation, overfitting reduction, and performance improvement. |

### Next Steps
- Take an intermediate deep learning with PyTorch course.
- Strengthen probability, statistics, linear algebra, and calculus.
- Pick a dataset and train a neural network end-to-end.


## Chapter Summary

| Concept | Key Point |
|---|---|
| `TensorDataset` | Pairs features and labels. |
| `DataLoader` | Batches and optionally shuffles data. |
| Training loss | Measures optimization progress on training data. |
| Validation loss | Measures generalization during tuning. |
| Accuracy | Measures classification correctness. |
| Dropout | Randomly zeros activations during training. |
| Weight decay | Penalizes large parameters. |
| Hyperparameter search | Tests learning rates and other choices systematically. |

✅ **PDF coverage:** animals dataset, feature/target extraction, `TensorDataset`, `DataLoader`, train/validation/test splits, loss tracking, validation mode, overfitting, torchmetrics accuracy, dropout, weight decay, data augmentation, performance improvement steps, grid search, random search, course wrap-up, and next steps.
